# Bonus 1 — Visualize Your Results 📊
Interactive dashboard for sentiment distribution, products, review lengths, review dates, and complaint themes. Uses Task 2 `datasets/clustered_reviews.csv`. Sentiment here is **rating-derived** (1–2 negative, 3 neutral, 4–5 positive), not RoBERTa inference.

## 1. Imports and preparation

In [1]:
# !pip install -U pandas numpy plotly gradio scikit-learn
from pathlib import Path
import pandas as pd
import numpy as np
import plotly.express as px
import gradio as gr
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS
PATH = Path("datasets/clustered_reviews.csv")
if not PATH.exists(): raise FileNotFoundError(f"Run Task 2 first: {PATH}")
df = pd.read_csv(PATH, low_memory=False)
required = ["product_name", "kmeans_category", "reviews.text", "reviews.rating"]
missing = [x for x in required if x not in df.columns]
if missing: raise ValueError(f"Missing columns: {missing}")
df = df.dropna(subset=required).copy()
df["reviews.rating"] = pd.to_numeric(df["reviews.rating"], errors="coerce")
df = df[df["reviews.rating"].between(1, 5)].copy()
df["sentiment"] = np.select([df["reviews.rating"] <= 2, df["reviews.rating"] == 3], ["Negative", "Neutral"], default="Positive")
df["review_length"] = df["reviews.text"].astype(str).str.split().str.len()
df["product_name"] = df["product_name"].astype(str)
df["kmeans_category"] = df["kmeans_category"].astype(str)
date_col = next((c for c in ["reviews.date", "reviews.dateAdded", "date"] if c in df.columns), None)
df["review_month"] = pd.to_datetime(df[date_col], errors="coerce", utc=True).dt.strftime("%Y-%m") if date_col else np.nan
print(f"{len(df):,} reviews, {df.product_name.nunique()} products, {df.kmeans_category.nunique()} categories")
display(df.head())

55,611 reviews, 69 products, 6 categories


,id,product_name,name,brand,categories,source,reviews.rating,reviews.title,reviews.text,kmeans_cluster,kmeans_category,sentiment,review_length,review_month
0,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",1429_1.csv,5.0,Kindle,This product so far has not disappointed. My c...,0,Fire Tablets,Positive,27,NaN
1,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",1429_1.csv,5.0,very fast,great for beginner or experienced person. Boug...,0,Fire Tablets,Positive,14,NaN
2,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",1429_1.csv,5.0,Beginner tablet for our 9 year old son.,Inexpensive tablet for him to use and learn on...,0,Fire Tablets,Positive,26,NaN
3,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",1429_1.csv,4.0,Good!!!,I've had my Fire HD 8 two weeks now and I love...,0,Fire Tablets,Positive,117,NaN
4,AVqkIhwDv8e3D1O-lebb,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...","All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",Amazon,"Electronics,iPad & Tablets,All Tablets,Fire Ta...",1429_1.csv,5.0,Fantastic Tablet for kids,I bought this for my grand daughter when she c...,0,Fire Tablets,Positive,117,NaN


## 2. Overview and sentiment by category

In [6]:
order = ["Negative", "Neutral", "Positive"]
fig = px.bar(df.sentiment.value_counts().reindex(order, fill_value=0).rename_axis("sentiment").reset_index(name="reviews"), x="sentiment", y="reviews", color="sentiment", title="Sentiment distribution")
fig.show()
ct = pd.crosstab(df.kmeans_category, df.sentiment).reindex(columns=order, fill_value=0)
share = ct.div(ct.sum(axis=1), axis=0).mul(100).reset_index().melt(id_vars="kmeans_category", var_name="sentiment", value_name="percent")
px.bar(share, x="kmeans_category", y="percent", color="sentiment", barmode="stack", title="Sentiment share by category").show()

## 3. Review length and time

In [3]:
px.box(df, x="sentiment", y="review_length", color="sentiment", points=False, title="Review length by sentiment").show()
if df.review_month.notna().any():
    monthly = df.dropna(subset=["review_month"]).groupby(["review_month", "sentiment"]).size().reset_index(name="reviews")
    px.line(monthly, x="review_month", y="reviews", color="sentiment", title="Monthly review trends").show()
else:
    print("No usable review dates; monthly chart skipped")

No usable review dates; monthly chart skipped


## 4. Product statistics and complaint bigrams
Complaint phrases are candidate themes and should be verified against original reviews.

In [4]:
def product_stats(d):
    result = d.groupby("product_name").agg(reviews=("reviews.rating", "size"), average_rating=("reviews.rating", "mean"), negative=("sentiment", lambda x: (x == "Negative").sum())).reset_index()
    result["negative_pct"] = 100 * result.negative / result.reviews
    return result.sort_values(["reviews", "average_rating"], ascending=[False, False])

def complaint_themes(d, n=12):
    texts = d.loc[d.sentiment == "Negative", "reviews.text"].dropna().astype(str).drop_duplicates()
    if len(texts) < 2: return pd.DataFrame(columns=["theme", "mentions"])
    stop = list(ENGLISH_STOP_WORDS.union({"amazon", "product", "bought", "buy", "item", "really", "just", "like", "got"}))
    try:
        vec = CountVectorizer(stop_words=stop, ngram_range=(2, 2), min_df=2 if len(texts) >= 15 else 1)
        mat = vec.fit_transform(texts)
        counts = np.asarray((mat > 0).sum(axis=0)).ravel()
        ids = counts.argsort()[::-1][:n]
        return pd.DataFrame({"theme": vec.get_feature_names_out()[ids], "mentions": counts[ids]})
    except ValueError:
        return pd.DataFrame(columns=["theme", "mentions"])
display(product_stats(df).head(10))
display(complaint_themes(df))

,product_name,reviews,average_rating,negative,negative_pct
60,"Fire Tablet, 7 Display, Wi-Fi, 8 GB - Includes...",11227,4.455153,334,2.974971
32,AmazonBasics AAA Performance Alkaline Batterie...,8336,4.447817,803,9.632917
46,Echo (White),7001,4.656335,106,1.514069
31,AmazonBasics AA Performance Alkaline Batteries...,3724,4.453276,343,9.210526
25,Amazon Kindle Paperwhite - eBook reader - 4 GB...,3176,4.772355,30,0.944584
7,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2817,4.587149,54,1.916933
5,"All-New Fire HD 8 Tablet, 8 HD Display, Wi-Fi,...",2545,4.583890,53,2.082515
51,"Fire HD 8 Tablet with Alexa, 8 HD Display, 16 ...",2443,4.597217,44,1.801064
54,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",1692,4.524232,58,3.427896
55,"Fire Kids Edition Tablet, 7 Display, Wi-Fi, 16...",1663,4.530968,54,3.247144


,theme,mentions
0,don long,51
1,waste money,51
2,brand batteries,36
3,batteries don,35
4,aa batteries,33
5,battery life,32
6,worst batteries,30
7,bad batch,30
8,did work,27
9,year old,26


## 5. Interactive Gradio dashboard

In [5]:
categories = ["All categories"] + sorted(df.kmeans_category.dropna().unique().tolist())
def dashboard(category, minimum_reviews, top_n):
    d = df if category == "All categories" else df[df.kmeans_category == category]
    counts = d.sentiment.value_counts().reindex(["Negative", "Neutral", "Positive"], fill_value=0).rename_axis("sentiment").reset_index(name="reviews")
    sentiment_fig = px.pie(counts, names="sentiment", values="reviews", hole=.4, title="Sentiment distribution")
    p = product_stats(d); p = p[p.reviews >= minimum_reviews].head(top_n)
    product_fig = px.bar(p.sort_values("average_rating"), x="average_rating", y="product_name", orientation="h", title="Top products by average rating", range_x=[0, 5])
    themes = complaint_themes(d, top_n)
    theme_fig = px.bar(themes.sort_values("mentions"), x="mentions", y="theme", orientation="h", title="Frequent negative-review phrases")
    length_fig = px.histogram(d, x="review_length", color="sentiment", nbins=40, title="Review lengths by sentiment")
    if len(d): length_fig.update_xaxes(range=[0, max(20, min(250, float(d.review_length.quantile(.98))))])
    summary = f"### {len(d):,} reviews · {d.product_name.nunique()} products · {d['reviews.rating'].mean():.2f}/5 average rating" if len(d) else "No reviews for this selection"
    return summary, sentiment_fig, product_fig, theme_fig, length_fig, p.round(2)
with gr.Blocks(title="Amazon Review Analytics — Bonus 1") as app:
    gr.Markdown("# 📊 Amazon Review Analytics\nExplore sentiment, product ratings, review lengths, and complaints.")
    with gr.Row():
        category = gr.Dropdown(choices=categories, value="All categories", label="Category")
        minimum = gr.Slider(1, 100, value=5, step=1, label="Minimum reviews per product")
        top_n = gr.Slider(5, 20, value=10, step=1, label="Top products/themes")
    summary = gr.Markdown()
    with gr.Row():
        sentiment_chart = gr.Plot()
        products_chart = gr.Plot()
    with gr.Row():
        complaint_chart = gr.Plot()
        length_chart = gr.Plot()
    products_table = gr.Dataframe(interactive=False)
    btn = gr.Button("Update dashboard", variant="primary")
    inputs = [category, minimum, top_n]
    outputs = [summary, sentiment_chart, products_chart, complaint_chart, length_chart, products_table]
    btn.click(dashboard, inputs=inputs, outputs=outputs)
    app.load(dashboard, inputs=inputs, outputs=outputs)
app.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://c15522f8e750fd888e.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## 6. Optional Task 4 integration
Add this dashboard as a fourth Gradio tab to Task 4, or deploy it separately. The sentiment shown in this bonus dashboard is **derived from ratings** and must not be described as Task 1 model predictions. Product ratings are descriptive, not the weighted ranking used by Task 3. The dashboard requires `datasets/clustered_reviews.csv`.